# Visualizing the doop benchmark

Builds the doop points-to analysis program and renders it via the
`application/vnd.srdatalog.viz+json` mime type. With the
`srdatalog-viz` Jupyter labextension installed, the rule graph
appears inline below each cell. Without the extension, the cell
shows the `text/plain` fallback.

Doop is the largest benchmark in the suite — 78 rules, 74 relations,
96 runner variants once recursive rules expand into delta-seeded
versions for semi-naive evaluation.

## 1. Build the program

`doop.py` is auto-translated from the Nim source. The builder takes
a `meta` dict — those are the dataset-resolved integer constants
(class IDs, method IDs) pre-interned by `prepare_vpt_dataset.dl`.
Different inputs (batik, biojava, eclipse) get different metas;
the rule structure is identical across them.

In [ ]:
import json
from pathlib import Path

import doop

META_PATH = Path('../../SRDatalog/integration_tests/examples/doop/batik_meta.json')
meta = json.loads(META_PATH.read_text())
prog = doop.build_doopdb_program(meta)
len(prog.rules), len(prog.relations)

## 2. Default visualization (no JIT C++)

Leaving `prog` as the last expression of a cell triggers
`Program._repr_mimebundle_`. The default omits the per-rule generated
C++ kernels — on doop that's the difference between a 300 KB cell
output and a 3 MB one. The renderer still has the full HIR / MIR /
rule summary, which is what the graph view needs.

In [ ]:
prog

## 3. Full visualization (with JIT)

When you want to inspect generated kernels — typically for
performance debugging — call `prog.show(include_jit=True)`. The
renderer then has access to a `jit` map keyed by runner name
(e.g. `VarPointsTo_D0`, `VarPointsTo_D1`, etc. for recursive rules)
with the full C++ source for each.

In [ ]:
prog.show(include_jit=True)

## 4. Drill into a single rule

The bundle is a regular Python dict — you can slice it directly
without going through a renderer. Useful for debugging the rule's
compiled plan.

In [ ]:
from srdatalog.viz.bundle import get_visualization_bundle

bundle = get_visualization_bundle(prog, include_jit=True)
[r for r in bundle['rules'] if 'VarPointsTo' in r['heads']][:3]

In [ ]:
# Generated kernel C++ for one runner — first 1500 chars
name = next(iter(bundle['jit']))
print(f'=== {name} ===')
print(bundle['jit'][name][:1500])